# Sales by Time Period ETL

## Purpose
Provide real-time sales analysis by period of day (morning, afternoon, evening, night) to identify optimal business hours and customer behavior patterns.

## Input
* **Source:** `big_data.silver.orders` 
* **Source:** `big_data.silver.order_products` 
* **Source:** `big_data.silver.products_enriched` 

## Output
* **Target:** `big_data.gold.vw_sales_by_time_period`
* **Refresh:** Real-time (always reflects current Silver data)

## SQL Logic
1. JOIN orders with order_products on order_id
2. JOIN with products_enriched to get prices
3. GROUP BY period_of_day (morning, afternoon, evening, night)
4. Calculate metrics: total_orders, total_items, revenue, avg_price, reorder_rate
5. ORDER BY period

In [0]:
%sql
-- Sales by Time Period View
-- Purpose: Analyze sales patterns across 4 time periods

CREATE OR REPLACE VIEW big_data.gold.vw_sales_by_time_period AS
SELECT 
  o.period_of_day,
  COUNT(DISTINCT o.order_id) AS total_orders,
  COUNT(op.product_id) AS total_items,
  ROUND(SUM(p.price_usd), 2) AS estimated_revenue_usd,
  ROUND(AVG(p.price_usd), 2) AS avg_item_price_usd,
  ROUND(SUM(CASE WHEN op.reordered THEN 1 ELSE 0 END) * 100.0 / COUNT(op.product_id), 2) AS reorder_rate
FROM big_data.silver.orders o
JOIN big_data.silver.order_products op ON o.order_id = op.order_id
JOIN big_data.silver.products_enriched p ON op.product_id = p.product_id
GROUP BY o.period_of_day
ORDER BY o.period_of_day;

In [0]:
%sql
-- Verify view exists and preview all time periods
-- Returns 4 rows (morning, afternoon, evening, night)

SELECT * FROM big_data.gold.vw_sales_by_time_period;